# Extract All Results — GPU Server
Scans **every** results folder, deduplicates, computes AUC / Mean-Last-5 / Best,
shows a coverage gap table, and prints the numbers the paper claims so you can
verify them before running new experiments.

Run top-to-bottom on the GPU server (`~/FedLLM-Re/rework/`).

In [ ]:
import json, os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)

plt.rcParams.update({'figure.dpi': 130, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

# ── root of rework directory ───────────────────────────────────────────────────
ROOT = os.path.expanduser('~/FedLLM-Re/rework')
print('ROOT:', ROOT)
print('exists:', os.path.exists(ROOT))

In [ ]:
# ── discover every results folder ─────────────────────────────────────────────
DATASET_SUBFOLDERS = {
    'yelp':   ['yelp'],
    'gsm8k':  ['gsm8k', 'gsmk'],
    'alpaca': ['alpaca'],
}

PRIMARY_METRIC = {
    'yelp':   'accuracy',
    'gsm8k':  'exact_match',
    'alpaca': 'rouge_l',
}

LABELS = {
    'homo_r8':    'Homo r=8',
    'hetero_pad': 'Hetero-Pad',
    'flexlora':   'FlexLoRA',
    'hetlora':    'HetLoRA',
    'hetlora_m':  'FedMoLoRA',
}

# The 5 methods that appear in Table I
PAPER_METHODS = ['homo_r8', 'hetero_pad', 'flexlora', 'hetlora', 'hetlora_m']

# Older / irrelevant names — excluded from all analysis
EXCLUDE_METHODS = {'spa_m', 'hetero_spa', 'homo_r4'}

# Seeds required per alpha / dataset (from GPU_TODO.md)
REQUIRED_SEEDS = {
    ('yelp',   0.5):  [42, 43, 44, 45, 46],
    ('yelp',   0.1):  [42, 43, 44, 45, 46],
    ('yelp',   0.01): [42, 43, 44, 45, 46],   # 40 rounds
    ('gsm8k',  0.5):  [42, 43, 44, 45, 46],
    ('gsm8k',  0.1):  [42, 43, 44],
    ('gsm8k',  0.01): [42, 43, 44, 45, 46],   # 40 rounds
    ('alpaca', 0.5):  [42, 43, 44],
}

def discover_json_files():
    """Find every .json result file under ROOT/results*."""
    all_files = []
    for top in sorted(glob.glob(os.path.join(ROOT, 'results*'))):
        all_files += glob.glob(os.path.join(top, '*.json'))
        all_files += glob.glob(os.path.join(top, '*', '*.json'))
    return sorted(set(all_files))

all_json = discover_json_files()
print(f'Total JSON files found: {len(all_json)}')
from collections import Counter
folders = Counter(os.path.dirname(f).replace(ROOT+'/', '') for f in all_json)
for folder, count in sorted(folders.items()):
    print(f'  {folder}: {count} files')

In [ ]:
# ── infer dataset from file path ───────────────────────────────────────────────
def infer_dataset(filepath, data):
    if 'dataset' in data:
        return data['dataset']
    path_lower = filepath.lower()
    if 'alpaca' in path_lower:
        return 'alpaca'
    if 'gsm8k' in path_lower or 'gsmk' in path_lower:
        return 'gsm8k'
    if 'yelp' in path_lower:
        return 'yelp'
    rounds = data.get('rounds', [])
    if rounds:
        r0 = rounds[0]
        if 'rouge_l' in r0:     return 'alpaca'
        if 'exact_match' in r0: return 'gsm8k'
        if 'accuracy' in r0:    return 'yelp'
    return 'unknown'

# ── load all files into a tidy DataFrame ──────────────────────────────────────
rows = []
skipped = []
seen = set()

for fp in all_json:
    try:
        with open(fp) as f:
            data = json.load(f)
    except Exception as e:
        skipped.append((fp, str(e))); continue

    method  = data.get('method', 'unknown')
    seed    = data.get('seed', -1)
    alpha   = data.get('alpha', -1)
    dataset = infer_dataset(fp, data)
    metric_key = PRIMARY_METRIC.get(dataset)

    # Skip excluded / legacy methods
    if method in EXCLUDE_METHODS:
        continue

    if metric_key is None:
        skipped.append((fp, f'unknown dataset {dataset}')); continue

    dedup_key = (dataset, method, alpha, seed)
    if dedup_key in seen:
        continue
    seen.add(dedup_key)

    round_data = data.get('rounds', [])
    if not round_data:
        skipped.append((fp, 'no rounds data')); continue

    for r in round_data:
        val = r.get(metric_key)
        if val is None:
            continue
        rows.append({
            'dataset': dataset,
            'method':  method,
            'alpha':   float(alpha),
            'seed':    int(seed),
            'round':   int(r['round']),
            'metric':  float(val),
            'source':  os.path.dirname(fp).replace(ROOT+'/', ''),
        })

df = pd.DataFrame(rows)
print(f'Loaded {len(df)} round-level rows from {len(seen)} unique runs')
print(f'Methods present: {sorted(df["method"].unique())}')
if skipped:
    print(f'Skipped {len(skipped)} files')
    for fp, reason in skipped[:5]:
        print(f'  {reason}: {os.path.basename(fp)}')

In [ ]:
# ── per-run summary statistics ─────────────────────────────────────────────────
def compute_stats(df):
    """Compute AUC, Mean-Last-5, Best per (dataset, method, alpha, seed)."""
    records = []
    for (ds, method, alpha, seed), g in df.groupby(['dataset', 'method', 'alpha', 'seed']):
        vals = g.sort_values('round')['metric'].values
        if len(vals) == 0:
            continue
        auc    = float(np.mean(vals))
        ml5    = float(np.mean(vals[-5:])) if len(vals) >= 5 else float(np.mean(vals))
        best   = float(np.max(vals))
        nround = len(vals)
        records.append({
            'dataset': ds, 'method': method, 'alpha': alpha, 'seed': seed,
            'auc': auc, 'mean_l5': ml5, 'best': best, 'n_rounds': nround,
        })
    return pd.DataFrame(records)

stats_per_run = compute_stats(df)

def aggregate_stats(stats_per_run):
    """Mean ± std across seeds."""
    records = []
    for (ds, method, alpha), g in stats_per_run.groupby(['dataset', 'method', 'alpha']):
        records.append({
            'dataset':      ds,
            'method':       method,
            'alpha':        alpha,
            'auc_mean':     g['auc'].mean(),
            'auc_std':      g['auc'].std(ddof=1) if len(g) > 1 else 0.0,
            'ml5_mean':     g['mean_l5'].mean(),
            'ml5_std':      g['mean_l5'].std(ddof=1) if len(g) > 1 else 0.0,
            'best_mean':    g['best'].mean(),
            'best_std':     g['best'].std(ddof=1) if len(g) > 1 else 0.0,
            'n_seeds':      len(g),
            'seeds':        sorted(g['seed'].tolist()),
            'n_rounds_avg': g['n_rounds'].mean(),
        })
    return pd.DataFrame(records)

agg = aggregate_stats(stats_per_run)
print('Aggregated stats shape:', agg.shape)
agg.head()

In [ ]:
# ── COVERAGE GAP TABLE ─────────────────────────────────────────────────────────
# Shows exactly what seeds exist and what's still needed for each
# (dataset, method, alpha) combination required by the paper.

print('=' * 80)
print('COVERAGE AUDIT — what exists vs. what the paper needs')
print('=' * 80)

needs_run = []  # list of (dataset, method, alpha, missing_seeds) to queue

for (dataset, alpha), req_seeds in sorted(REQUIRED_SEEDS.items()):
    print(f'\n[{dataset.upper()}  α={alpha}]  need seeds={req_seeds}')
    for m in PAPER_METHODS:
        sub = agg[(agg['dataset'] == dataset) & (agg['method'] == m) & (agg['alpha'] == alpha)]
        if sub.empty:
            have = []
        else:
            have = sub.iloc[0]['seeds']
        missing = [s for s in req_seeds if s not in have]
        status = 'OK' if not missing else f'MISSING {missing}'
        label = LABELS.get(m, m)
        auc_str = ''
        if not sub.empty:
            r = sub.iloc[0]
            auc_str = f"  AUC={r['auc_mean']:.3f}±{r['auc_std']:.3f}  (n={r['n_seeds']} seeds)"
        print(f'  {label:<15} have={have}  → {status}{auc_str}')
        if missing:
            needs_run.append({'dataset': dataset, 'method': m, 'alpha': alpha,
                              'missing_seeds': missing})

print('\n' + '=' * 80)
print(f'TOTAL GAPS: {len(needs_run)} (dataset, method, alpha) combinations need more seeds')
print('=' * 80)

In [ ]:
# ── PAPER NUMBERS CROSS-CHECK ──────────────────────────────────────────────────
# Compare current computed values vs. what is written in submission_v4.tex

PAPER_CLAIMS = [
    # (dataset, method, alpha, metric, paper_value, description)
    ('yelp',   'hetlora_m', 0.1,  'auc',  0.446, 'FedMoLoRA Yelp α=0.1 AUC → 44.6%'),
    ('yelp',   'hetlora',   0.1,  'auc',  0.415, 'HetLoRA Yelp α=0.1 Raw AUC (Table IV) → 41.5%'),
    ('yelp',   'hetlora_m', 0.01, 'ml5',  0.424, 'FedMoLoRA Yelp α=0.01 ML5 → 42.4%'),
    ('yelp',   'hetlora',   0.01, 'ml5',  0.368, 'HetLoRA Yelp α=0.01 ML5 → 36.8%'),
    ('gsm8k',  'hetlora_m', 0.5,  'ml5',  0.7647,'FedMoLoRA GSM8K α=0.5 ML5 → 76.47%'),
    ('gsm8k',  'hetlora_m', 0.1,  'ml5',  0.7693,'FedMoLoRA GSM8K α=0.1 ML5 → 76.93%'),
    ('gsm8k',  'hetlora_m', 0.01, 'ml5',  0.739, 'FedMoLoRA GSM8K α=0.01 ML5 → 73.9%'),
    ('gsm8k',  'hetlora',   0.01, 'ml5',  0.740, 'HetLoRA GSM8K α=0.01 ML5 → 74.0%'),
]

print('=' * 80)
print('PAPER CLAIMS CROSS-CHECK')
print('=' * 80)
for dataset, method, alpha, metric_col, paper_val, desc in PAPER_CLAIMS:
    sub = agg[(agg['dataset'] == dataset) & (agg['method'] == method) & (agg['alpha'] == alpha)]
    if sub.empty:
        col_key = 'auc_mean' if metric_col == 'auc' else 'ml5_mean'
        print(f'  ❌ NO DATA   {desc}')
        continue
    r = sub.iloc[0]
    col_key = 'auc_mean' if metric_col == 'auc' else 'ml5_mean'
    current = r[col_key]
    diff = current - paper_val
    n = int(r['n_seeds'])
    flag = '✅' if abs(diff) < 0.005 else ('⚠️ ' if abs(diff) < 0.02 else '❌')
    print(f'  {flag} n={n}  paper={paper_val:.3f}  got={current:.3f}  Δ={diff:+.3f}  | {desc}')

In [ ]:
# ── FULL AUC TABLE (all datasets) ─────────────────────────────────────────────

def fmt(mean, std, n):
    if np.isnan(mean):
        return '—'
    stars = '' if n >= 5 else f'(n={n})'
    return f'{mean*100:.1f}±{std*100:.1f} {stars}'.strip()

for dataset in ['yelp', 'gsm8k', 'alpaca']:
    sub = agg[agg['dataset'] == dataset].copy()
    if sub.empty:
        print(f'\n{dataset.upper()}: no data'); continue
    alphas = sorted(sub['alpha'].unique())
    methods_present = [m for m in PAPER_METHODS if m in sub['method'].values]
    
    print(f'\n{"="*70}')
    print(f'{dataset.upper()}  — AUC (% mean across rounds, mean±std across seeds)')
    print(f'{"="*70}')
    header = f'{"Method":<16}' + ''.join(f' α={a:<8}' for a in alphas)
    print(header)
    print('-' * len(header))
    for m in methods_present + [mm for mm in sub['method'].unique() if mm not in methods_present]:
        row_str = f'{LABELS.get(m, m):<16}'
        for alpha in alphas:
            s = sub[(sub['method'] == m) & (sub['alpha'] == alpha)]
            if s.empty:
                row_str += f' {"—":<10}'
            else:
                r = s.iloc[0]
                row_str += f' {fmt(r["auc_mean"], r["auc_std"], r["n_seeds"]):<10}'
        print(row_str)
    
    print(f'\n{dataset.upper()}  — Mean-Last-5')
    print('-' * len(header))
    for m in methods_present + [mm for mm in sub['method'].unique() if mm not in methods_present]:
        row_str = f'{LABELS.get(m, m):<16}'
        for alpha in alphas:
            s = sub[(sub['method'] == m) & (sub['alpha'] == alpha)]
            if s.empty:
                row_str += f' {"—":<10}'
            else:
                r = s.iloc[0]
                row_str += f' {fmt(r["ml5_mean"], r["ml5_std"], r["n_seeds"]):<10}'
        print(row_str)

In [ ]:
# ── WHAT TO RUN NEXT (priority queue) ─────────────────────────────────────────
import pandas as pd

gap_df = pd.DataFrame(needs_run)
if gap_df.empty:
    print('No gaps! All required runs are complete.')
else:
    # Priority: α=0.01 first (hardest / highest value), then by dataset
    priority_alpha = {0.01: 0, 0.1: 1, 0.5: 2}
    priority_dataset = {'yelp': 0, 'gsm8k': 1, 'alpaca': 2}
    gap_df['_pa'] = gap_df['alpha'].map(priority_alpha)
    gap_df['_pd'] = gap_df['dataset'].map(priority_dataset)
    gap_df = gap_df.sort_values(['_pa', '_pd']).drop(columns=['_pa', '_pd'])
    gap_df['label'] = gap_df['method'].map(LABELS)
    gap_df['n_missing'] = gap_df['missing_seeds'].apply(len)
    gap_df['total_runs'] = gap_df['n_missing']  # 1 run per seed
    
    print(f'RUNS NEEDED ({gap_df["total_runs"].sum()} total):')
    print(gap_df[['dataset', 'label', 'alpha', 'missing_seeds', 'n_missing']].to_string(index=False))
    
    print('\n── Suggested run commands (adapt script name as needed) ──')
    for _, row in gap_df.iterrows():
        seeds_str = ' '.join(map(str, row['missing_seeds']))
        rounds = 40 if row['alpha'] == 0.01 else 20
        print(f'python experiments/run_{row["dataset"]}.py '
              f'--method {row["method"]} '
              f'--alpha {row["alpha"]} '
              f'--seeds {seeds_str} '
              f'--rounds {rounds}')

In [ ]:
# ── CONVERGENCE CURVES (whatever data exists) ──────────────────────────────────
COLORS = {
    'homo_r8':    '#888888',
    'hetero_pad': '#4e79a7',
    'flexlora':   '#f28e2b',
    'hetlora':    '#e15759',
    'hetlora_m':  '#9467bd',
}

for dataset in ['yelp', 'gsm8k']:
    sub_df = df[df['dataset'] == dataset]
    if sub_df.empty:
        print(f'No {dataset} data for curves.'); continue
    alphas = sorted(sub_df['alpha'].unique())
    fig, axes = plt.subplots(1, len(alphas), figsize=(5 * len(alphas), 4), sharey=False)
    if len(alphas) == 1:
        axes = [axes]
    for ax, alpha in zip(axes, alphas):
        for m in PAPER_METHODS:
            g = sub_df[(sub_df['method'] == m) & (sub_df['alpha'] == alpha)]
            if g.empty:
                continue
            pivot = g.pivot_table(index='round', columns='seed', values='metric')
            rounds = pivot.index.values
            mu  = pivot.mean(axis=1).values
            std = pivot.std(axis=1).fillna(0).values
            color = COLORS.get(m, '#aaaaaa')
            lw = 2.2 if m == 'hetlora_m' else 1.5
            ax.plot(rounds, mu, label=LABELS.get(m, m), color=color, lw=lw)
            ax.fill_between(rounds, mu - std, mu + std, alpha=0.12, color=color)
        ax.set_title(f'{dataset.upper()}  α={alpha}')
        ax.set_xlabel('Round')
        ax.set_ylabel('Accuracy' if dataset == 'yelp' else 'Exact Match')
        ax.legend(fontsize=7)
    plt.suptitle(f'{dataset.upper()} Convergence (available data)', fontweight='bold')
    plt.tight_layout()
    out = f'figures/{dataset}_convergence_current.png'
    os.makedirs('figures', exist_ok=True)
    plt.savefig(out, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out}')

In [ ]:
# ── SAVE EXTRACTED DATA TO CSV ─────────────────────────────────────────────────
# Useful for reloading without re-scanning all files

os.makedirs('results_extracted', exist_ok=True)

df.to_csv('results_extracted/all_rounds.csv', index=False)
stats_per_run.to_csv('results_extracted/stats_per_run.csv', index=False)
agg.to_csv('results_extracted/stats_aggregated.csv', index=False)

print('Saved:')
print('  results_extracted/all_rounds.csv        — every round of every run')
print('  results_extracted/stats_per_run.csv     — AUC/ML5/Best per (method, alpha, seed)')
print('  results_extracted/stats_aggregated.csv  — mean±std across seeds')